### Динамика троса

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

A = 0.4       # амплитуда, м
f = 0.25      # частота, Гц
g = 9.81

omega = 2 * np.pi * f
t = np.linspace(0, 8, 2000)

x = A * np.sin(omega * t)
v = A * omega * np.cos(omega * t)
a = -A * omega**2 * np.sin(omega * t)

v_max = A * omega
a_max = A * omega**2
theta_deg = np.degrees(np.arctan(a_max / g))

print(f"Пиковая скорость: {v_max:.3f} м/с")
print(f"Пиковое ускорение: {a_max:.3f} м/с²")
print(f"Наклон для ускорения: {theta_deg:.2f}°")

fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)

for ax, y, label in zip(
    axes, [x, v, a],
    ["Положение, м", "Скорость, м/с", "Ускорение, м/с²"]
):
    ax.plot(t, y)
    ax.set_ylabel(label)
    ax.grid(True)

axes[-1].set_xlabel("Время, с")
plt.tight_layout()
plt.show()


### Тяговооруженость и ветер

In [ ]:

from math import sqrt

m = 10.0
g = 9.81
rho = 1.225

# Предварительные допущения, заменить данными аппарата
Cd = 1.0
S = 0.15  # эффективная площадь, м²
V_w = 10.0
a_x = 0.987  # пиковое ускорение троса, м/с²

weight = m * g
F_w = 0.5 * rho * Cd * S * V_w**2

# Упрощённый худший случай: ветер и требуемое
# ускорение увеличивают потребную боковую силу
T_req = sqrt(weight**2 + (m * a_x + F_w)**2)

K_required = T_req / weight

# Пример инженерного запаса, не нормативное значение
reserve = 1.3
K_design = reserve * K_required

print(f"Вес: {weight:.1f} Н")
print(f"Оценка силы ветра: {F_w:.1f} Н")
print(f"Расчётная тяга: {T_req:.1f} Н")
print(f"Тяговооружённость без запаса: {K_required:.2f}")
print(f"Предварительная тяговооружённость: {K_design:.2f}")


### Точность позиционирования

In [ ]:

import numpy as np

v_max = 0.4 * 2 * np.pi * 0.25
tolerance = 0.02

# Пример распределения бюджета, метры
errors = {
    "Измерение троса": 0.005,
    "Оценка БВС": 0.005,
    "Прогноз цели": 0.008,
    "Контур управления": 0.010,
}

worst_case = sum(errors.values())
rss = np.sqrt(sum(e**2 for e in errors.values()))

print(f"Сумма по худшему случаю: {worst_case*1000:.1f} мм")
print(f"RSS: {rss*1000:.1f} мм")

# Исследование бюджета задержки
for allocated_error_mm in [2, 5, 10, 20]:
    e = allocated_error_mm / 1000
    tau = e / v_max
    print(
        f"Бюджет {allocated_error_mm:2d} мм -> "
        f"задержка не более {tau*1000:.1f} мс"
    )


### Вибрации при взлёте

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

A = 0.005  # амплитуда перемещения, м
frequencies = np.linspace(10, 40, 301)

omega = 2 * np.pi * frequencies
a_peak = A * omega**2

for f in [10, 20, 30, 40]:
    a = A * (2 * np.pi * f)**2
    print(f"{f:2d} Гц: {a:.1f} м/с² ({a/9.81:.1f} g)")

plt.plot(frequencies, a_peak)
plt.xlabel("Частота, Гц")
plt.ylabel("Пиковое ускорение, м/с²")
plt.grid(True)
plt.tight_layout()
plt.show()


### Управление и отказы

In [ ]:

from enum import Enum, auto


class State(Enum):
    APPROACH = auto()
    CONTACT_CHECK = auto()
    SECURING = auto()
    SECURED = auto()
    RELEASE_CHECK = auto()
    RELEASING = auto()
    FREE_FLIGHT_CHECK = auto()
    DEPARTURE = auto()
    ABORT = auto()
    FAULT = auto()


def next_state(state, s):
    """s — словарь проверенных признаков и команд."""

    if s.get("critical_fault", False):
        return State.FAULT

    if state == State.APPROACH:
        if not s.get("relative_pose_valid", False):
            return State.ABORT
        if s.get("contact_confirmed", False):
            return State.CONTACT_CHECK

    elif state == State.CONTACT_CHECK:
        if s.get("support_verified", False):
            return State.SECURING
        if s.get("contact_timeout", False):
            return State.ABORT

    elif state == State.SECURING:
        if s.get("lock_confirmed", False):
            return State.SECURED

    elif state == State.SECURED:
        if s.get("release_authorized", False):
            return State.RELEASE_CHECK

    elif state == State.RELEASE_CHECK:
        if s.get("departure_conditions_ok", False):
            return State.RELEASING

    elif state == State.RELEASING:
        if s.get("release_confirmed", False):
            return State.FREE_FLIGHT_CHECK

    elif state == State.FREE_FLIGHT_CHECK:
        if s.get("free_flight_confirmed", False):
            return State.DEPARTURE

    return state
